# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [24]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
from datetime import datetime, timezone
import os
load_dotenv(override=True)

True

In [25]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)
load_time= datetime.now()
print(f"Loaded at: {load_time}")   

Loaded at: 2026-08-16 17:56:10.840874


In [26]:
google_api_key=os.getenv("GOOGLE_API_KEY")
GoogleAI = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=google_api_key)

In [27]:
# Some lists!

checklist = []
completed = []

In [29]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [30]:
get_checklist_report()

''

In [31]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [32]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [33]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [34]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [35]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [36]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [37]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [38]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [42]:
def loop(messages):
    response = GoogleAI.chat.completions.create(model="gemini-3.6-flash", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = GoogleAI.chat.completions.create(model="gemini-3.6-flash", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [46]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A container has 80 litres of pure milk. 8 litres of milk is taken out and replaced with water. This process is done a second time. What is the final quantity of pure milk left in the container?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [47]:
checklist, completed = [], []
loop(messages)

Checklist #1: Identify initial parameters from the problem
Checklist #2: Calculate the quantity of milk remaining after the first operation
Checklist #3: Calculate the quantity of milk remaining after the second operation
Checklist #4: Verify the result using the replacement formula and state the final answer

Identified initial parameters:
- Initial volume of pure milk (V) = 80 litres
- Volume of liquid removed and replaced with water (x) = 8 litres
- Number of operations (n) = 2

Checklist #1: Identify initial parameters from the problem
Checklist #2: Calculate the quantity of milk remaining after the first operation
Checklist #3: Calculate the quantity of milk remaining after the second operation
Checklist #4: Verify the result using the replacement formula and state the final answer

First operation calculations:
- Milk removed = 8 litres
- Remaining milk = 80 - 8 = 72 litres
- Water added = 8 litres
- Total mixture volume = 80 litres
- Proportion of milk in mixture = 72 / 80 = 0.9 (90%)

Checklist #1: Identify initial parameters from the problem
Checklist #2: Calculate the quantity of milk remaining after the first operation
Checklist #3: Calculate the quantity of milk remaining after the second operation
Checklist #4: Verify the result using the replacement formula and state the final answer

Second operation calculations:
- Mixture removed = 8 litres
- Milk removed in 8 litres of mixture = 8 * 0.9 = 7.2 litres
- Final milk remaining = 72 - 7.2 = 64.8 litres

Checklist #1: Identify initial parameters from the problem
Checklist #2: Calculate the quantity of milk remaining after the first operation
Checklist #3: Calculate the quantity of milk remaining after the second operation
Checklist #4: Verify the result using the replacement formula and state the final answer

Verified using the formula:
Final Milk = V * (1 - x/V)^n
Final Milk = 80 * (1 - 8/80)^2 = 80 * (0.9)^2 = 80 * 0.81 = 64.8 litres.

Checklist #1: Identify initial parameters from the problem
Checklist #2: Calculate the quantity of milk remaining after the first operation
Checklist #3: Calculate the quantity of milk remaining after the second operation
Checklist #4: Verify the result using the replacement formula and state the final answer

To find the final quantity of pure milk left in the container, we can analyze the process step-by-step or use the 
standard mixture replacement formula.

Step-by-Step Breakdown:

1. Initial State:
   • Pure milk = 80 litres
   • Water = 0 litres

2. First Replacement:
   • Milk removed = 8 litres
   • Milk remaining = 80 - 8 = 72 litres
   • Water added = 8 litres
   • Concentration of milk in mixture = 72 / 80 = 0.9 (or 90%)

3. Second Replacement:
   • Volume of mixture removed = 8 litres
   • Pure milk removed = 8 litres × 0.9 = 7.2 litres
   • Final milk remaining = 72 - 7.2 = 64.8 litres

---

Verification using the Standard Formula:

Final Quantity = Initial Quantity × (1 - x / V)^n

Where:
• V = Initial volume = 80 litres
• x = Volume replaced each time = 8 litres
• n = Number of operations = 2

Calculation:
• Final Quantity = 80 × (1 - 8 / 80)^2
• Final Quantity = 80 × (0.9)^2
• Final Quantity = 80 × 0.81 = 64.8 litres

---

Final Answer:
The final quantity of pure milk left in the container is 64.8 litres.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>